In [1]:
import os
import cv2
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    jaccard_score,
    classification_report
)
import time

In [2]:
TRAIN_DIR = "Dataset/train"
VAL_DIR = "Dataset/validation"
TEST_DIR = "Dataset/test"

# number of pixels to sample per image
PIXELS_PER_IMAGE = 5000
#PIXELS_PER_IMAGE = 122500

FEATURE_MODES = {
    "RGB": ["r", "g", "b"],
    "RGB+HSV": ["r", "g", "b", "h", "s", "v"],
    "RGB+HSV+Texture": ["r", "g", "b", "h", "s", "v", "local_mean", "local_var"],
    "RGB+HSV+Texture+VegIdx": ["r", "g", "b", "h", "s", "v", "local_mean", "local_var", "exg", "ndi", "veg", "exg_blur3", "exg_blur7"],
}

In [3]:
def load_image(path):
    img = cv2.imread(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def load_mask(path):
    mask = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    # 0 (black) -> plant 1
    # 255 (white) -> background 0
    return (mask < 127).astype(np.uint8) # Convert grayscale mask into binary mask

def extract_features(image, mode=None):
    rgb = image.astype(np.float32) # convert image to float32
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV).astype(np.float32) # convert RGB to HSV
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY).astype(np.float32) # brightness channel for local variance calc

    R = rgb[:,:,0]
    G = rgb[:,:,1]
    B = rgb[:,:,2]
    
    epsilon = 1e-6
    
    # green plant features
    exg = 2*G - R - B # excess green
    ndi = (G-R) /(G+R+epsilon) # normalise difference index 
    veg = G / (R**0.667 * B**0.333 + epsilon) # vegetative index 

    local_mean = cv2.blur(gray, (5, 5)) # get avg brightness of 5x5 area
    local_var = cv2.blur(gray**2, (5, 5)) - local_mean**2  # compute variance of brightness (high var=mixed plants/soil)

    # blur exg channel at 3px and 7px 
    exg_blur3 = cv2.GaussianBlur(exg, (3,3), 0)
    exg_blur7 = cv2.GaussianBlur(exg, (7,7), 0)

    all_features = {
        "r": rgb[:,:,0], 
        "g": rgb[:,:,1], 
        "b": rgb[:,:,2],
        "h": hsv[:,:,0], 
        "s": hsv[:,:,1], 
        "v": hsv[:,:,2],
        "exg": exg, 
        "ndi": ndi, 
        "veg": veg,
        "exg_blur3": exg_blur3, 
        "exg_blur7": exg_blur7,
        "local_mean": local_mean, 
        "local_var": local_var,
    }

    # Return everything if no mode selected
    if mode is not None:
        keys = mode
    else:
        keys = list(all_features.keys())
    
    return np.dstack([all_features[k] for k in keys])


# n_samples = number of samples we wanna train on
# 35% plant - 65% background
def sample_pixels(features, mask, n_samples, plant_ratio=0.35):
    h, w, f = features.shape
    
    # flatten 
    X = features.reshape(-1, f)
    y = mask.reshape(-1)
    
    plant_idx = np.where(y == 1)[0]
    bg_idx = np.where(y == 0)[0]

    n_plant = min(int(n_samples * plant_ratio), len(plant_idx))
    n_bg = min(n_samples - n_plant, len(bg_idx))

    if n_plant == 0:
        print("err no plants found")
        return X[:n_samples], y[:n_samples]

    # randomely choose plant and background pixels
    chosen = np.concatenate([
        np.random.choice(plant_idx, n_plant, replace=False),
        np.random.choice(bg_idx, n_bg, replace=False),
    ])
    # get rid of ordering bias
    np.random.shuffle(chosen)
    return X[chosen], y[chosen]

def build_dataset(folder, pixels_per_image=None, feature_keys=None):
    X_all, y_all = [], []
     
    for filename in sorted(os.listdir(folder)):
        if "_mask" in filename:
            continue
 
        img_path  = os.path.join(folder, filename)
        mask_path = os.path.join(folder, filename.replace(".png", "_mask.png"))

        image = load_image(img_path)
        mask  = load_mask(mask_path)
 
        features = extract_features(image, feature_keys)
 
        X, y = sample_pixels(features, mask, pixels_per_image)
 
        X_all.append(X)
        y_all.append(y)
        print(f" Done {filename}")
 
    return np.vstack(X_all), np.concatenate(y_all)


# return predicted binary mask for image
def predict_mask(model, image, feature_keys=None):
    features = extract_features(image, feature_keys)
    h, w, f = features.shape
    
    # run RF
    y_pred = model.predict(features.reshape(-1, f))
    
    return y_pred.reshape(h, w).astype(np.uint8)


# returns comparison of true mask vs predicted
def evaluate_mask(gt_mask, pred_mask):
    y_true = gt_mask.reshape(-1)
    y_pred = pred_mask.reshape(-1)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "iou": jaccard_score(y_true, y_pred, zero_division=0),
    }
                                            
def evaluate_on_dir(model, directory, feature_keys=None, verbose=True):
    all_metrics = []
 
    for filename in sorted(os.listdir(directory)):
        if "_mask" in filename:
            continue
 
        img_path  = os.path.join(directory, filename)
        mask_path = img_path.replace(".png", "_mask.png")
 
        image = load_image(img_path)
        gt_mask = load_mask(mask_path)
        pred = predict_mask(model, image, feature_keys)
        metrics = evaluate_mask(gt_mask, pred)
        all_metrics.append(metrics)
 
        if verbose:
            print(
                f"  {filename} | "
                f"A: {metrics['accuracy']:.4f}  "
                f"P: {metrics['precision']:.4f}  "
                f"R: {metrics['recall']:.4f}  "
                f"F1: {metrics['f1']:.4f}  "
                f"IoU: {metrics['iou']:.4f}"
            )
 
    return {
        "accuracy": np.mean([m["accuracy"] for m in all_metrics]),
        "precision": np.mean([m["precision"] for m in all_metrics]),
        "recall": np.mean([m["recall"] for m in all_metrics]),
        "f1": np.mean([m["f1"] for m in all_metrics]),
        "iou": np.mean([m["iou"] for m in all_metrics]),
    }

def run_experiment(config_name, feature_keys):
    print(f"\n")
    print(f" {config_name}")
 
    print("\n Creating training dataset (1/3)")
    
    X_train, y_train = build_dataset(TRAIN_DIR, PIXELS_PER_IMAGE, feature_keys)
    
    print(f"  {X_train.shape}")
 
    print("\n Training Random Forest (2/3)")
    model = RandomForestClassifier(
        n_estimators=150,
        max_depth=20,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1,
    )
    
    train_start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - train_start
 
    print("\n Evaluating (3/3)")
    test_start = time.time()
    mean_metrics = evaluate_on_dir(model, VAL_DIR, feature_keys, verbose=True)
    test_time = time.time() - test_start
 
    print(f"\n  --- Mean Validation Results ({config_name}) ---")
    print(f"  Accuracy: {mean_metrics['accuracy']:.4f}")
    print(f"  Precision: {mean_metrics['precision']:.4f}")
    print(f"  Recall: {mean_metrics['recall']:.4f}")
    print(f"  F1: {mean_metrics['f1']:.4f}")
    print(f"  IoU: {mean_metrics['iou']:.4f}")
    print(f"  Training Time: {train_time:.2f} s")
    print(f"  Testing Time:  {test_time:.2f} s")
 
    return {"config": config_name, **mean_metrics, "train_time": train_time, "test_time": test_time}

def main():
    np.random.seed(42)
 
    all_results = []
    for config_name, feature_keys in FEATURE_MODES.items():
        result = run_experiment(config_name, feature_keys)
        all_results.append(result)
 
    # Final comparison table
    col = 30
    print(f"\n\n{'='*110}")
    print("  FEATURE CONFIGURATION COMPARISON")
    print(f"{'='*110}")
    print(
        f"  {'Config':<{col}} "
        f"{'Accuracy':>10} {'Precision':>10} {'Recall':>8} {'F1':>8} {'IoU':>8} "
        f"{'Train(s)':>10} {'Test(s)':>10}"
    )
    print(
        f"  {'-'*col} "
        f"{'-'*10} {'-'*10} {'-'*8} {'-'*8} {'-'*8} "
        f"{'-'*10} {'-'*10}"
    )
    for r in all_results:
        print(
            f"  {r['config']:<{col}} "
            f"{r['accuracy']:>10.4f} "
            f"{r['precision']:>10.4f} "
            f"{r['recall']:>8.4f} "
            f"{r['f1']:>8.4f} "
            f"{r['iou']:>8.4f}"
            f"{r['train_time']:>10.2f} "
            f"{r['test_time']:>10.2f}"
        )
    print(f"{'='*110}\n")
 
 
if __name__ == "__main__":
    main()




 RGB

 Creating training dataset (1/3)
 Done FPWW0220011_RGB1_20180316_100219_6.png
 Done FPWW0220032_RGB1_20171213_141430_6.png
 Done FPWW0220032_RGB1_20180112_121036_6.png
 Done FPWW0220032_RGB1_20180124_115802_6.png
 Done FPWW0220032_RGB1_20180207_120354_6.png
 Done FPWW0220032_RGB1_20180220_142617_6.png
 Done FPWW0220032_RGB1_20180305_114520_6.png
 Done FPWW0220032_RGB1_20180316_100627_6.png
 Done FPWW0220032_RGB1_20180323_143114_6.png
 Done FPWW0220032_RGB1_20180406_110138_6.png
 Done FPWW0220032_RGB1_20180411_113950_6.png
 Done FPWW0220032_RGB1_20180419_113813_6.png
 Done FPWW0220032_RGB1_20180424_112157_6.png
 Done FPWW0220066_RGB1_20180207_121054_6.png
 Done FPWW0220071_RGB1_20180124_120540_6.png
 Done FPWW0220078_RGB1_20171213_142438_6.png
 Done FPWW0220078_RGB1_20180112_122034_6.png
 Done FPWW0220078_RGB1_20180124_120700_6.png
 Done FPWW0220078_RGB1_20180207_121309_6.png
 Done FPWW0220078_RGB1_20180220_143501_6.png
 Done FPWW0220078_RGB1_20180305_115417_6.png
 Done FPWW0220